# Quantization (binning): turning counts into buckets

_Feature Engineering_

**Chop a heavy-tailed count into a handful of bins so the big values stop dominating.**

Some raw numbers span a huge range. A business on Yelp might have 3 reviews or
       3,000. A song might have been played twice or two million times. When a feature like this
       feeds a model, the giant values shout over the small ones: a linear model treats the
       difference between 1 and 2 reviews the same as between 2,001 and 2,002, even though the first
       difference matters far more.

       Quantization (also called binning) fixes this. You stop using the raw count and
       instead say which bucket it falls into: "0&ndash;9 reviews", "10&ndash;99 reviews", and so
       on. The model now sees a small ordered category instead of an unbounded number. This is the very
       first transform in Chapter 2 of Zheng & Casari's Feature Engineering for Machine Learning,
       and their running example is the Yelp business review_count &mdash; a classic
       heavy-tailed count.

---

This notebook is a practice scaffold for the **Quantization (binning): turning counts into buckets** lesson. We build the idea up one step at a time: see the problem a single straight line runs into, then watch binning fix it. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

### Step 1 — Build a non-monotonic dataset

Binning shines when the target is **not** a straight-line function of the feature. So we build one continuous feature `x` in [0, 10] and make `y = 1` only inside a *middle band* (3 < x < 7), with a little label noise. The signal lives in the middle, not at the ends — a shape no single line can trace. We split into train and test before doing anything else.

In [ ]:
# LESSON: Quantization / binning — bucket a continuous feature, then one-hot the buckets.
# PROBLEM: a LINEAR model fits ONE straight line through a raw feature. If the target is
# HIGH in the MIDDLE of x and LOW at both ends (non-monotonic), a single line cannot
# separate the classes -> near-chance accuracy.
# FIX: cut x into quantile buckets and one-hot encode them, so the model can learn one
# weight PER bucket and say "middle buckets = class 1".

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(0)              # fixed seed -> reproducible numbers
n = 600
x = rng.uniform(0, 10, size=n)              # one continuous feature in [0, 10]

# Target is 1 in the middle band (3 < x < 7), else 0.
in_middle_band = (x > 3) & (x < 7)
y = in_middle_band.astype(int)

# Add 5% random label flips (realistic noise).
flip = rng.uniform(size=n) < 0.05
y = np.where(flip, 1 - y, y)

X = x.reshape(-1, 1)                         # sklearn wants a 2D feature matrix
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

### Step 2 — Reproduce the problem with raw `x`

A logistic regression on the raw feature is a single monotone S-curve in `x`: it can only say "y rises (or falls) as x grows". It has no way to pick out a *middle* band. We fit it, score it on the test set, and overlay its predicted P(y=1) on the raw scatter so you can see the curve flatten out and fit almost nothing.

In [ ]:
# Fit a logistic model on the RAW feature and score it.
raw_clf = LogisticRegression()
raw_clf.fit(X_tr, y_tr)
raw_acc = accuracy_score(y_te, raw_clf.predict(X_te))

# Plot the raw data (left panel) and the model's monotone predicted curve.
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(x, y + rng.normal(0, 0.02, n), s=8, alpha=0.4, label="data (y jittered)")
ax[0].set_title("STEP 2: raw x vs y — class 1 lives in the MIDDLE")
ax[0].set_xlabel("x")
ax[0].set_ylabel("y (0/1)")

grid = np.linspace(0, 10, 200).reshape(-1, 1)
grid_probs = raw_clf.predict_proba(grid)[:, 1]
ax[0].plot(grid, grid_probs, "r-", lw=2, label=f"raw linear fit (acc={raw_acc:.3f})")
ax[0].legend(loc="upper right")

### Step 3 — Apply the fix: quantile-bin and inspect the buckets

`KBinsDiscretizer` with `strategy="quantile"` cuts `x` into 8 equal-count buckets, and `encode="onehot-dense"` turns each bucket into its own 0/1 column. We fit the bin edges on the **training set only** (no leakage), then look at the per-bucket class rate: the middle buckets should sit near 1, the edge buckets near 0 — exactly the structure the raw line missed.

In [ ]:
# Cut x into 8 equal-count quantile buckets, one-hot encoded.
binner = KBinsDiscretizer(n_bins=8, encode="onehot-dense", strategy="quantile")
X_tr_bin = binner.fit_transform(X_tr)       # fit buckets on TRAIN only (no leakage)
X_te_bin = binner.transform(X_te)

# Which bucket did each of the original points fall into?
bucket_id = binner.transform(X).argmax(axis=1)
n_bins = X_tr_bin.shape[1]

# Per-bucket class rate: middle buckets should be ~1, edge buckets ~0.
rates = []
for b in range(n_bins):
    in_bucket = bucket_id == b
    rate = y[in_bucket].mean() if np.any(in_bucket) else 0
    rates.append(rate)

ax[1].bar(range(n_bins), rates, color="steelblue")
ax[1].set_title("STEP 4: per-bucket class rate (middle buckets ~ 1)")
ax[1].set_xlabel("quantile bucket")
ax[1].set_ylabel("rate of y=1")
ax[1].set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

### Step 4 — Show the fix pays off

Now we fit the *same* logistic regression, but on the binned one-hot features. Because each bucket has its own column, the model learns one independent weight per bucket and can simply switch the middle buckets "on". We print the before/after accuracy: the raw fit hovered near chance, while the binned fit jumps well above it.

In [ ]:
# SAME LogisticRegression, now on the BINNED one-hot features.
bin_clf = LogisticRegression()
bin_clf.fit(X_tr_bin, y_tr)
bin_acc = accuracy_score(y_te, bin_clf.predict(X_te_bin))

print(f"PROBLEM (raw x):       accuracy = {raw_acc:.3f}")
print(f"FIX (quantile-binned): accuracy = {bin_acc:.3f}")
print(f"PROBLEM (raw): {raw_acc:.3f}   →   FIX (engineered): {bin_acc:.3f}")

## Reference implementation — pandas + numpy

### Step 1 — Load the heavy-tailed Yelp counts

The book's running example is the Yelp business `review_count` — a classic heavy-tailed count where most businesses have a handful of reviews and a few have thousands. We load the businesses from the dataset's JSON file into a DataFrame so we can experiment with different binning schemes on a real skewed column.

In [ ]:
import numpy as np
import pandas as pd
import json

# --- Load the Yelp business data (Yelp Dataset Challenge, round 6) ---
# Get it from the book's repo: github.com/alicezheng/feature-engineering-book
biz_file = open('yelp_academic_dataset_business.json')
biz_df = pd.DataFrame([json.loads(x) for x in biz_file.readlines()])
biz_file.close()

# 'review_count' is a heavy-tailed count: most businesses have a few reviews,
# a handful have thousands.

### Step 2 — Fixed-width binning (linear vs exponential)

The simplest scheme is **fixed-width**: divide the count by a constant width with integer division, so 0–9 → bin 0, 10–19 → bin 1, and so on. But on a heavy tail that leaves huge gaps, so the chapter also shows **exponential** binning — group by powers of 10 by taking `log10` and flooring — which gives each order of magnitude its own bin.

In [ ]:
# === Fixed-width binning ===
# Linear: map a count to a bin by INTEGER DIVISION (bin width = 10).
small_counts = np.array([0, 5, 13, 28, 37, 99, 100, 7000])
linear_bins = np.floor_divide(small_counts, 10)
# -> array([  0,   0,   1,   2,   3,   9,  10, 700])

# Exponential: group by powers of 10 (take the log10, then floor).
large_counts = np.array([296, 8286, 64011, 80, 3, 725, 867, 2215,
                         7689, 11495, 91897, 44, 28, 7971, 926, 122, 22222])
exponential_bins = np.floor(np.log10(large_counts))
# -> array([2., 3., 4., 1., 0., 2., 2., 3., 3., 4., 4., 1., 1., 3., 2., 2., 4.])

### Step 3 — Quantile binning (adaptive, equal-count)

Quantile binning is **adaptive**: it places the cut points at the data's own percentiles so every bucket holds the same number of rows, no matter how skewed the counts are. `pd.qcut(x, 4, labels=False)` gives the integer bin index. Reading the deciles straight off `review_count` shows the edges bunch tightly at the low end and then leap to 58 — exactly mirroring the heavy tail.

In [ ]:
# === Quantile binning (adaptive: equal-count bins) ===
# Cut review_count into 4 equal-count bins; labels=False -> integer bin index.
quantile_bins = pd.qcut(large_counts, 4, labels=False)

# Read the deciles straight off the data to see where the edges fall:
decile_levels = [.1, .2, .3, .4, .5, .6, .7, .8, .9]
deciles = biz_df['review_count'].quantile(decile_levels)
print(deciles)
# For the Yelp review_count the deciles came out:
# 0.1 -> 3   0.2 -> 4   0.3 -> 5   0.4 -> 6   0.5 -> 8
# 0.6 -> 12  0.7 -> 17  0.8 -> 28  0.9 -> 58
# Tight at the low end, then jumping to 58 -- exactly mirroring the heavy tail.

## Visualize the data & results

_Take a skewed real count (the 'mean area' of cells in load_breast_cancer) and split it into 5 fixed-width bins vs 5 quantile bins. Which scheme keeps the bins evenly populated?_

### Step 1 — Grab a real skewed feature

The `mean area` column of `load_breast_cancer` is a right-skewed, count-like measurement — a good stand-in for the Yelp tail on a dataset that ships with Colab. We pull it out and subsample 60 sorted values so the bin occupancy is easy to eyeball.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer

d = load_breast_cancer()                         # 569 real cell-nucleus measurements
fi = list(d.feature_names).index('mean area')
area = d.data[:, fi]                             # right-skewed, count-like feature

# Subsample to 60 sorted values so bin occupancy is easy to read.
rng = np.random.RandomState(0)
sample_idx = rng.choice(len(area), 60, replace=False)
area = np.sort(area[sample_idx])

### Step 2 — Fixed-width vs quantile occupancy

We cut the same 60 values two ways: 5 **equal-width** bins (edges spaced evenly across the range) and 5 **quantile** bins (edges at the 0/20/40/60/80/100 percentiles). Counting how many points land in each bin shows the contrast: fixed-width bins come out lopsided on a skewed feature, while quantile bins stay evenly populated.

In [ ]:
# Fixed-width: 5 equal-width bins across the range (numpy only, no pandas).
fw_edges = np.linspace(area.min(), area.max(), 6)
fw_occ = np.histogram(area, bins=fw_edges)[0]
print('fixed-width edges  :', np.round(fw_edges).astype(int))
print('fixed-width counts :', fw_occ)            # -> [17 25  7  5  6]  (lopsided)

# Quantile: edges at the 0,20,40,60,80,100 percentiles -> equal-count bins.
q_edges = np.percentile(area, [0, 20, 40, 60, 80, 100])
q_occ = np.histogram(area, bins=q_edges)[0]
print('quantile edges     :', np.round(q_edges).astype(int))
print('quantile counts    :', q_occ)             # -> [12 12 12 12 12]  (even)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You bin the Yelp review_count into 20 equal-width bins. Almost every business lands in bin 0 and bins 5 through 19 are empty. What went wrong and what are the two fixes from the chapter?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize the feature is heavy-tailed: a few businesses have thousands of reviews, but the median has a handful. — _Equal-width bins of width $w$ put nearly all the mass in the first bin and leave the tail bins empty._
- Switch to exponential bins: $\lfloor\log_{10}x\rfloor$ via np.floor(np.log10(...)). — _Power-of-10 bands are ten times wider each step, so the long tail gets a few wide bins instead of hundreds of empty ones._
- Or switch to quantile bins via pd.qcut, cutting at the deciles. — _Equal-count bins guarantee every bucket is populated, regardless of how skewed the raw counts are._

**Answer:** The empty-bin problem: equal-width bins over a heavy tail dump everyone into bin 0 and leave the rest empty. The chapter's two fixes are exponential (log) binning &mdash; np.floor(np.log10(x)) &mdash; and quantile binning &mdash; pd.qcut(x, n, labels=False), whose deciles for the Yelp data were 3, 4, 5, 6, 8, 12, 17, 28, 58.

</details>

**Problem 2.** A teammate computes quantile bin edges on the full dataset, then splits into train and test. Why is this a problem, and how should the edges be produced instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that quantile edges are learned from data &mdash; they are parameters of the transform. — _Unlike a fixed width of 10, the decile cut points come from the values themselves._
- Computing them on the full dataset uses test-set values to set the edges. — _That is data leakage: information from the test set has bled into the feature definition, inflating measured performance._
- Fit the quantiles on the training set only, store the edges, and apply those same edges to validation/test/production. — _This keeps the transform honest and reproducible, exactly like fitting any other preprocessor on train only._

**Answer:** Quantile edges are learned parameters, so computing them on all the data leaks test information into the feature. Fit the deciles on the training set only, save those edges, and reuse them everywhere. Fixed-width edges (a constant width like 10) don't have this issue because they don't depend on the sample.

</details>

**Problem 3.** For two different features &mdash; a 1-to-5 star rating, and a play-count that ranges from 2 to 4 million &mdash; which binning scheme fits each, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Look at the range and shape of each feature. — _The star rating is small and bounded; the play-count is heavy-tailed across orders of magnitude._
- For the bounded, roughly-uniform rating, use fixed-width linear bins (or leave it as is). — _Equal-width bins are simple and the buckets stay populated because the range is small and even._
- For the play-count, use exponential (log) bins or quantile bins. — _Equal-width bins would be almost all empty; log bands or equal-count quantiles match the heavy tail._

**Answer:** The bounded, uniform-ish star rating suits fixed-width linear bins. The heavy-tailed play-count needs an adaptive scheme: exponential (log) binning for fixed, data-independent power-of-10 bands, or quantile binning for guaranteed equal-count buckets. Equal-width bins on the play-count would be mostly empty.

</details>